# 모스 (Mo's)

`-` 정적 배열에서 여러 개의 쿼리를 제곱근 분할법을 기반으로 정렬한 뒤 처리하는 알고리즘

## 수열과 쿼리 5

- 문제 출처: [백준 13547번](https://www.acmicpc.net/problem/13547)

`-` 일반적인 세그먼트 트리를 사용해서 풀기에는 난감하다

`-` 임의의 구간에 존재하는 서로 다른 수의 개수를 세야 하기에 노드별로 구간에 어떤 수가 있는지 저장해야 한다

`-` 하지만 이러면 취합하는 데 최악의 경우 $O(N)$의 시간 복잡도를 가지게 된다

`-` 어떤 방법이 있는지 공부하고 오자

`-` Mo's 알고리즘을 통해 해결할 수 있는 문제라고 한다

`-` 원소가 $N$개 있는데 이를 $P$개의 그룹으로 쪼갠다고 해보자

`-` 그럼 각 그룹엔 $\frac{N}{P}$개의 원소가 있다

`-` 그리고 각 그룹마다 어떤 수가 몇 개 있는지와 서로 다른 수의 개수를 저장하자

`-` 쿼리가 주어졌는데 범위가 첫 번째 그룹만을 가리킨다고 해보자

`-` 원래라면 $O\left(\frac{N}{P}\right)$의 시간 복잡도를 가지겠지만 여기선 $O(1)$에 처리할 수 있다

`-` 하지만 쿼리 범위가 항상 깔끔하게 주어지는 건 아니다

`-` 시간 복잡도는 그룹 내 원소의 개수와 그룹의 개수에 영향을 받는다

`-` 즉, $O\left(\max\left(\frac{N}{P}, P\right)\right)$이다

`-` 따라서 $\frac{N}{P}= P$를 만족해야 시간 복잡도가 최소이고 이때의 $P=\sqrt{N}$이다

`-` 이것이 제곱근 분할법이다

`-` 근데 이것만으론 문제를 해결할 수 없다

`-` 쿼리 범위가 좁으면 괜찮지만 넓다면 서로 다른 수의 개수를 세는 데 $O(N)$의 시간 복잡도를 가진다

`-` 잘 생각해보면 굳이 쿼리를 입력받은 순서대로 처리할 필요가 없다

`-` 출력만 입력받은 순서대로 하면 되는 것이다!

`-` 모든 $M$개의 쿼리에 대해 $j$를 포함하는 그룹의 인덱스를 기준으로 오름차순 정렬하자

`-` 만약 동일하다면 $i$를 기준으로 오름차순 정렬하자

`-` 이제 첫 번째 쿼리부터 처리해보자

`-` 처음엔 순회하며 원소를 세면 된다

`-` 이는 $O(N)$이다

`-` 하지만 그 다음 쿼리부턴 빠르게 처리할 수 있다

`-` $j$가 같은 그룹에 속하므로 이를 조정하는 건 $O\left(\sqrt{N}\right)$이다

`-` 이제 $i$를 조정해보자

`-` $j$가 같은 그룹에 속하는 모든 쿼리를 생각해보자

`-` $i$를 기준으로 오름차순 정렬했으므로 조정할 원소의 개수는 최대 $O(N)$이다

`-` 이는 $j$가 포함되는 그룹이 바뀔 때만 발생하므로 총 $O\left(N\sqrt{N}\right)$이다

`-` 만약 $j$가 속한 그룹이 바뀌어야 한다면 초기화하면 된다

`-` 초기화하면 다음 쿼리부턴 순회하며 원소를 세야 하지만 이는 그룹이 바뀔 때 $1$번만 발생하며 그룹의 개수는 $O\left(\sqrt{N}\right)$이다

`-` 따라서 전체 알고리즘의 시간 복잡도는 $O\left((N+M)\sqrt{N} + M\log M\right)$이다

`-` $\log M \le \sqrt{N}$일테니 $O\left((N+M)\sqrt{N}\right)$로 표현해도 괜찮다

`-` 참고: https://infossm.github.io/blog/2019/02/09/mo's-algorithm/

`-` 참고: https://infossm.github.io/blog/2021/07/19/distinct-value-query/

`-` 사실 그룹별로 최솟값, 구간 합 같은 통계량을 가지고 있을 필요는 없다

`-` 그래서 단순히 정렬된 쿼리마다 이전과 차이나는 만큼 원소를 추가하거나 제거하고 이를 쿼리 결과에 반영하면 된다

`-` 즉, 단순히 쿼리 순서만 바꿨을 뿐인데 시간 복잡도가 $O(NM)$에서  $O\left((N+M)\sqrt{N}\right)$으로 줄어든 것이다!

In [55]:
import math


def compress(array):
    unique = set(array)
    mapping = {a: i for i, a in enumerate(unique)}
    return [mapping[a] for a in array]


def mos(array, quries):
    global RESULT
    n = len(array)
    m = len(quries)
    array = compress(array)
    block_size = math.ceil(n**0.5)
    quries.sort(key=lambda q: (q[1] // block_size, q[0]))
    counts = [0] * n
    answers = [0] * m
    RESULT = 0
    i_prev, j_prev = 0, -1
    for i, j, k in quries:
        while i < i_prev:
            i_prev -= 1
            add(array[i_prev], counts)
        while j > j_prev:
            j_prev += 1
            add(array[j_prev], counts)
        while i > i_prev:
            remove(array[i_prev], counts)
            i_prev += 1
        while j < j_prev:
            remove(array[j_prev], counts)
            j_prev -= 1
        i_prev, j_prev = i, j
        answers[k] = RESULT
    return answers


def add(a, counts):
    global RESULT
    if counts[a] == 0:
        RESULT += 1
    counts[a] += 1


def remove(a, counts):
    global RESULT
    counts[a] -= 1
    if counts[a] == 0:
        RESULT -= 1


def solution():
    N = int(input())
    array = list(map(int, input().split()))
    M = int(input())
    quries = []
    for k in range(M):
        i, j = map(int, input().split())
        quries.append((i - 1, j - 1, k))
    answers = mos(array, quries)
    print("\n".join(map(str, answers)))


solution()

# input
# 5
# 1 1 2 1 3
# 3
# 1 5
# 2 4
# 3 5

 5
 1 1 2 1 3
 3
 1 5
 2 4
 3 5


3
2
3


`-` [서로 다른 수와 쿼리 1](https://www.acmicpc.net/problem/14897) 날먹 ㄱㄱ

`-` [민호의 소원](https://www.acmicpc.net/problem/13028)이랑 [Poklon](https://www.acmicpc.net/problem/14413)도 같이 ㄱㄱ

## 배열의 힘

- 문제 출처: [백준 8462번](https://www.acmicpc.net/problem/8462)

`-` Mo's 알고리즘을 사용하는 문제이다

`-` 자연수 $s$에 대해 현재의 $K_s$를 $x$라고 하자

`-` 자연수별로 $x$를 추적하기 위해 $O(\max(a))$ 크기의 배열을 선언하자

`-` 딕셔너리를 사용하면 $O(N)$이지만 $N$과 $\max(a)$의 차이가 작으면 오히려 더 느리다

`-` 이제 쿼리마다 부분 배열의 힘이 이전 쿼리에 비해 얼마나 변화하는지 따져보자

`-` $x$가 기존보다 $1$ 증가하면 새로운 부분 배열의 힘 중 $x$가 차지하는 비중은 $(x+1)^2 s$가 된다

`-` 기존엔 $x^2 s$이었므로 변화량은 $(2x+1)s$이다

`-` 반대로 $x$가 기존보다 $1$ 감소하면 새로운 부분 배열의 힘 중 $x$가 차지하는 비중은 $(x-1)^2 s$가 된다

`-` 그럼 변화량은 $(-2x+1)s$이다

`-` 포인터의 위치가 바뀔때마다 부분 배열의 힘의 변화량을 $O(1)$에 반영해주면 된다

`-` 따라서 전체 알고리즘의 시간 복잡도는 $O\left((n+t) \sqrt{n} + \max(a)\right)$이다

In [2]:
import math


def mos(array, quries):
    global RESULT
    n = len(array)
    m = len(quries)
    block_size = math.ceil(n**0.5)
    quries.sort(key=lambda q: (q[1] // block_size, q[0]))
    max_value = max(array)
    counts = [0] * (max_value + 1)
    answers = [0] * m
    RESULT = 0
    i_prev, j_prev = 0, -1
    for i, j, k in quries:
        while i < i_prev:
            i_prev -= 1
            add(array[i_prev], counts)
        while j > j_prev:
            j_prev += 1
            add(array[j_prev], counts)
        while i > i_prev:
            remove(array[i_prev], counts)
            i_prev += 1
        while j < j_prev:
            remove(array[j_prev], counts)
            j_prev -= 1
        i_prev, j_prev = i, j
        answers[k] = RESULT
    return answers


def add(a, counts):
    global RESULT
    RESULT += (2 * counts[a] + 1) * a 
    counts[a] += 1


def remove(a, counts):
    global RESULT
    RESULT -= (2 * counts[a] - 1) * a 
    counts[a] -= 1


def solution():
    n, t = map(int, input().split())
    array = list(map(int, input().split()))
    quries = []
    for k in range(t):
        i, j = map(int, input().split())
        quries.append((i - 1, j - 1, k))
    answers = mos(array, quries)
    print("\n".join(map(str, answers)))


solution()

# input
# 8 3
# 4 3 1 1 1 3 1 2
# 2 7
# 1 6
# 3 8

 8 3
 4 3 1 1 1 3 1 2
 2 7
 1 6
 3 8


28
25
21


## 수열과 쿼리 6

- 문제 출처: [백준 13548번](https://www.acmicpc.net/problem/13548)

`-` Mo's 알고리즘을 사용하는 문제인 건 알겠는데 어떻게 접근할지 모르겠다

`-` 그래서 태그 봤는데 그냥 Mo's 알고리즘만 사용하는 문제이다

`-` $x$가 최빈값이라 하자

`-` $x$의 빈도수가 증가하거나 $x$ 외의 수들의 빈도수가 증가하는 건 처리하기 괜찮다

`-` 이건 $O(1)$에 최빈값의 빈도수를 알 수 있다

`-` 문제는 $x$의 빈도수가 감소했을 때이다

`-` 그럼 최빈값이 바뀔 수 있다

`-` 하지만 최빈값이 무엇인지 확인하는 건 $O(N)$이다

`-` 현재 최빈값이 $x$이고 $x$의 빈도수를 $F_x$라 하자

`-` 빈도수가 $F_x$인 수가 $x$ 뿐이면 최빈값의 빈도수는 $F_x-1$이 된다

`-` 하지만 그렇지 않다면 최빈값의 빈도수는 여전히 $F_x$이다

`-` 그럼 $F_x$를 빈도수로 가지는 수들이 몇 개 있는지 추적하면 되는 거 아닌가?

`-` 빈도수별로 원소들의 개수를 매핑해서 가지고 있자!

`-` 수열의 원소별로 빈도수도 매핑해서 가지고 있자

`-` 이건 빈도수별로 원소들의 개수가 매핑된 딕셔너리에 자기 자신의 빈도수가 몇인지 빠르게 찾기 위해서이다

`-` 어떤 수의 빈도수가 증가, 감소함에 따라 두 개의 매핑 딕셔너리와 최빈값의 빈도수를 관리하면 된다

`-` 이제 Mo's 알고리즘으로 $O\left((N+M)\sqrt{N}\right)$에 문제를 해결할 수 있다

- 시간 복잡도 표현

`-` 원소별 빈도수를 저장하기 위해 $\max(A)$ 크기의 배열을 선언해야 된다

`-` 그러면 이를 시간 복잡도에 반영해야 되는 거 아닌가라고 생각할 수 있다

`-` 어차피 좌표 압축하면 $N$에 바운딩돼서 생략했다 (물론 하진 않았다, 어차피 최댓값 $10^5$밖에 안됨)

`-` 그리고 애초에 $10^5$으로 고정시키면 $O(1)$이다

`-` 끝!

In [1]:
import math


def mos(array, quries):
    global RESULT
    n = len(array)
    m = len(quries)
    block_size = math.ceil(n**0.5)
    quries.sort(key=lambda q: (q[1] // block_size, q[0]))
    max_value = max(array)
    num2frequency = [0] * (max_value + 1)
    frequency2count = [0] * (n + 1)
    answers = [0] * m
    RESULT = 0
    i_prev, j_prev = 0, -1
    for i, j, k in quries:
        while i < i_prev:
            i_prev -= 1
            add(array[i_prev], num2frequency, frequency2count)
        while j > j_prev:
            j_prev += 1
            add(array[j_prev], num2frequency, frequency2count)
        while i > i_prev:
            remove(array[i_prev], num2frequency, frequency2count)
            i_prev += 1
        while j < j_prev:
            remove(array[j_prev], num2frequency, frequency2count)
            j_prev -= 1
        i_prev, j_prev = i, j
        answers[k] = RESULT
    return answers


def add(a, num2frequency, frequency2count):
    global RESULT
    f = num2frequency[a]
    num2frequency[a] += 1
    frequency2count[f] -= 1
    frequency2count[f + 1] += 1
    if f + 1 > RESULT:
        RESULT = f + 1


def remove(a, num2frequency, frequency2count):
    global RESULT
    f = num2frequency[a]
    num2frequency[a] -= 1
    frequency2count[f] -= 1
    frequency2count[f - 1] += 1
    if f == RESULT and frequency2count[f] == 0:
        RESULT = f - 1


def solution():
    N = int(input())
    array = list(map(int, input().split()))
    M = int(input())
    quries = []
    for k in range(M):
        i, j = map(int, input().split())
        quries.append((i - 1, j - 1, k))
    answers = mos(array, quries)
    print("\n".join(map(str, answers)))


solution()

# input
# 5
# 1 2 1 3 3
# 3
# 1 3
# 2 3
# 1 5

 5
 1 2 1 3 3
 3
 1 3
 2 3
 1 5


2
1
2


`-` [Frequent values](https://www.acmicpc.net/problem/6515) 문제랑 [화려한 마을3](https://www.acmicpc.net/problem/12999) 문제 날먹 ㄱㄱ

## 수열과 쿼리 4

- 문제 출처: [백준 13546번](https://www.acmicpc.net/problem/13546)

`-` [수열과 쿼리 6](https://www.acmicpc.net/problem/13548) 문제보다 조금 더 어려운 버전이다

`-` 현재 두 포인터가 가리키는 구간 내 원소별 인덱스를 덱으로 관리할 것이다

`-` 예컨대 쿼리 구간이 오른쪽으로 옮겨져 $a_i$가 추가됐다고 해보자

`-` 이는 현재 쿼리 구간 내에서 여러 $a_i$ 중 가장 오른쪽에서 등장한 것이다

`-` $a_i$의 인덱스를 관리하는 덱에 $i$를 추가하고 첫 번째 원소와 마지막 원소의 차이를 계산하자

`-` 이것이 현재 추적하고 있는 쿼리 결괏값보다 크다면 갱신하자

`-` 추가는 괜찮은데 문제는 $a_i$가 제거될 때 발생한다

`-` 오른쪽 포인터가 왼쪽으로 움직여서 $a_i$가 제거됐다고 해보자

`-` 그럼 덱에서 $i$를 제거할 것이다

`-` 이로 인해 쿼리 결괏값이 감소할 수도 있다

`-` 그럼 가능한 쿼리 결괏값 중 최댓값을 계산해야 되는데 이를 단순히 원소별로 순회해서 찾으면 $O(N)$이다

`-` 이를 빠르게 찾기 위해 구간 최댓값 세그먼트 트리를 사용하자

`-` 원소의 값을 인덱스로 가지며 노드에는 해당 원소의 덱의 첫 번째 값과 마지막 값의 차이를 저장할 것이다

`-` 그럼 가능한 쿼리 결괏값 중 최댓값을 $O(\log N)$에 찾을 수 있다

`-` 문제는 시간 복잡도가 $O\left((N+M)\sqrt{N}\log N\right)$이라 최적화 없이는 파이썬으로 통과하기 힘들다는 것이다

`-` 나는 다음의 $3$가지 최적화를 수행했다

`-` 첫 번째로 세그먼트 트리를 재귀가 아닌 비재귀로 구현했다 (챗지피티의 도움을 받았다)

`-` 두 번째로 쿼리를 정렬할 때 지그재그 정렬을 사용했다 (홀수 블럭은 내림차순, 짝수 블럭은 오름차순)

`-` 세 번째로 쿼리 구간이 바뀌어 영향을 받은 원소들을 모아둔 뒤 마무리로 한 번에 갱신해줬다 (같은 원소를 중복 갱신하는 비효율을 막음)

`-` 제한 시간이 $14$초인데 실행 시간이 $13.276$초로 간신히 통과했다 ㅋㅋ

In [10]:
import math
from collections import deque


def mos(array, quries):
    n = len(array)
    m = len(quries)
    block_size = math.ceil(n**0.5)
    quries.sort(key=lambda q: (q[1] // block_size, q[0] if (q[1] // block_size) % 2 == 0 else -q[0]))
    max_value = max(array)
    num2indices = [deque() for _ in range(max_value + 1)]
    tree, size = build(max_value)
    root = 1
    answers = [0] * m
    l_prev, r_prev = 0, -1
    changes = set()
    for l, r, k in quries:
        while l < l_prev:
            l_prev -= 1
            add_left(array[l_prev], l_prev, num2indices, changes)
        while r > r_prev:
            r_prev += 1
            add_right(array[r_prev], r_prev, num2indices, changes)
        while l > l_prev:
            remove_left(array[l_prev], num2indices, changes)
            l_prev += 1
        while r < r_prev:
            remove_right(array[r_prev], num2indices, changes)
            r_prev -= 1
        l_prev, r_prev = l, r
        for a in changes:
            if not num2indices[a]:
                update(a, 0, tree, size)
                continue
            max_gap = num2indices[a][-1] - num2indices[a][0]
            update(a, max_gap, tree, size)
        changes.clear()
        answers[k] = tree[root]
    return answers


def build(n):
    size = 1
    while size <= n:
        size *= 2
    tree = [0] * 2 * size
    return tree, size


def add_left(a, index, num2indices, changes):
    num2indices[a].appendleft(index)
    changes.add(a)


def add_right(a, index, num2indices, changes):
    num2indices[a].append(index)
    changes.add(a)


def remove_left(a, num2indices, changes):
    num2indices[a].popleft()
    changes.add(a)


def remove_right(a, num2indices, changes):
    num2indices[a].pop()
    changes.add(a)


def update(index, value, tree, size):
    index += size
    tree[index] = value
    while index > 1:
        index >>= 1
        if tree[2 * index] > tree[2 * index + 1]:
            tree[index] = tree[2 * index]
        else:
            tree[index] = tree[2 * index + 1]


def solution():
    N, K = map(int, input().split())
    array = list(map(int, input().split()))
    M = int(input())
    quries = []
    for k in range(M):
        l, r = map(int, input().split())
        quries.append((l - 1, r - 1, k))
    answers = mos(array, quries)
    print("\n".join(map(str, answers)))


solution()

# input
# 7 7
# 4 5 6 6 5 7 4
# 5
# 6 6
# 5 6
# 3 5
# 3 7
# 1 7

 7 7
 4 5 6 6 5 7 4
 5
 6 6
 5 6
 3 5
 3 7
 1 7


0
0
1
1
6
